# Домашнее задание: VAE для латентных диффузионных моделей

## Обучение Variational Autoencoder в стиле Stable Diffusion

В ноутбуке все клетки должны выполняться без ошибок при последовательном их выполнении.


Темы: Архитектура VAE (ResBlock, Encoder, Decoder), Perceptual Loss и обучение, Adversarial Training (PatchGAN), Анализ латентного пространства

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import math
import copy
import random
import tarfile
from pathlib import Path
from typing import Optional, Tuple, List, Union

import numpy as np
import regex

from PIL import Image

import matplotlib
import matplotlib.pyplot as plt
import matplotlib_inline

import tqdm.autonotebook as tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.swa_utils import AveragedModel

try:
    from torch.optim.swa_utils import get_ema_multi_avg_fn
except ImportError:
    PARAM_LIST = Union[Tuple[torch.Tensor, ...], List[torch.Tensor]]
    def get_ema_multi_avg_fn(decay=0.999):
        @torch.no_grad()
        def ema_update(ema_param_list: PARAM_LIST, current_param_list: PARAM_LIST, _):
            if torch.is_floating_point(ema_param_list[0]) or torch.is_complex(ema_param_list[0]):
                torch._foreach_lerp_(ema_param_list, current_param_list, 1 - decay)
            else:
                for p_ema, p_model in zip(ema_param_list, current_param_list):
                    p_ema.copy_(p_ema * decay + p_model * (1 - decay))
        return ema_update

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid

import wandb

%matplotlib inline
matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg')

torch.backends.cudnn.benchmark = True

In [ ]:
def set_global_seed(seed: int) -> None:
    """Set global seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def check_numel(module: torch.nn.Module, params_numel: int, buffers_numel: Optional[int] = None) -> None:
    """Check whether module has correct number of parameters and buffers."""
    numel = sum(param.numel() for param in module.parameters())
    assert numel == params_numel, f'For params numel != correct numel: {numel} vs {params_numel}'
    if buffers_numel is not None:
        numel = sum(param.numel() for param in module.buffers())
        assert numel == buffers_numel, f'For buffers numel != correct numel: {numel} vs {buffers_numel}'


def hide_specs(module: torch.nn.Module) -> str:
    """Remove hyperparameters for model __repr__."""
    module_repr = str(module)
    for spec in regex.findall("\\(.*\\): (.*)", module_repr):
        module_repr = module_repr.replace(spec, regex.sub('\\(.*\\)', '(...)', spec))
    return module_repr

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Теория: VAE для латентных диффузий

## Мотивация

В работе [High-Resolution Image Synthesis with Latent Diffusion Models](https://arxiv.org/abs/2112.10752) (Rombach et al., 2022) предложена двухстадийная схема генерации изображений:
1. **Стадия 1:** обучить VAE, который сжимает изображение $x \in \mathbb{R}^{H \times W \times 3}$ в компактное латентное представление $z \in \mathbb{R}^{h \times w \times c}$, где $h = H/f$, $w = W/f$, а $f$ — фактор понижения разрешения.
2. **Стадия 2:** обучить диффузионную модель (DDPM / DDIM) работать в этом латентном пространстве.

В нашем случае: $H = W = 128$, $f = 8$, $h = w = 16$, $c = 4$. Это означает сжатие в $\frac{128 \times 128 \times 3}{16 \times 16 \times 4} = \frac{49152}{1024} = 48$ раз!

## Отличия от «учебного» VAE

В классическом VAE (Kingma & Welling, 2014) мы максимизируем ELBO:

$$\mathcal{L}_{\text{ELBO}} = \mathbb{E}_{q_\phi(z|x)} \big[\log p_\theta(x|z)\big] - D_{\text{KL}}\big(q_\phi(z|x) \| p(z)\big)$$

Для латентных диффузий подход существенно отличается:

### 1. Малый вес KL ($\beta \sim 10^{-6}$)

Нам **не нужно** сэмплировать красивые изображения из $p(z) = \mathcal{N}(0, I)$ — генерацией будет заниматься диффузионная модель. Нам нужно лишь слегка регуляризовать латентное пространство, чтобы оно было:
- **гладким** (близкие $z$ дают близкие $x$)
- **ограниченным** (значения $z$ не уходят в бесконечность)

### 2. Perceptual Loss (LPIPS)

Вместо попиксельного MSE, который даёт размытые реконструкции, используется **perceptual loss** — расстояние между фичами предобученной VGG сети:

$$\mathcal{L}_{\text{perc}} = \sum_l \lambda_l \left\| \phi_l(x) - \phi_l(\hat{x}) \right\|_1$$

где $\phi_l$ — признаки $l$-го слоя VGG.

### 3. Adversarial Loss (PatchGAN)

Для получения чётких текстур добавляется **PatchGAN дискриминатор** с hinge loss:

$$\mathcal{L}_D = \mathbb{E}\big[\text{ReLU}(1 - D(x))\big] + \mathbb{E}\big[\text{ReLU}(1 + D(\hat{x}))\big]$$
$$\mathcal{L}_G^{\text{adv}} = -\mathbb{E}\big[D(\hat{x})\big]$$

### Итоговый лосс генератора (encoder + decoder):

$$\mathcal{L}_G = \lambda_{\text{rec}} \cdot \| x - \hat{x} \|_1 + \lambda_{\text{perc}} \cdot \mathcal{L}_{\text{perc}} + \beta \cdot D_{\text{KL}} + \lambda_{\text{adv}} \cdot \mathcal{L}_G^{\text{adv}}$$

# Часть 0. Загрузка и предобработка данных

Мы будем использовать [ImageNette](https://github.com/fastai/imagenette) — подмножество ImageNet из 10 легко различимых классов (~13000 изображений). Мы ресайзим изображения до $128 \times 128$.

Классы ImageNette: tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute.

In [ ]:
# Скачиваем ImageNette (160px версию)
if not os.path.exists('./imagenette2-160'):
    if not os.path.exists('./imagenette2-160.tgz'):
        !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz
    with tarfile.open('./imagenette2-160.tgz', 'r:gz') as tar:
        tar.extractall('.')
    print('ImageNette extracted!')
else:
    print('ImageNette already exists!')

In [ ]:
class ImageNetteDataset(Dataset):
    """ImageNette dataset resized to img_size x img_size."""

    def __init__(self, root: str, split: str = 'train', img_size: int = 128):
        """
        Args:
            root: (str) Path to imagenette2-160 directory
            split: (str) 'train' or 'val'
            img_size: (int) Target image size
        """
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # scale to [-1, 1]
        ])

        split_dir = os.path.join(root, split)
        self.image_paths = []
        for class_dir in sorted(os.listdir(split_dir)):
            class_path = os.path.join(split_dir, class_dir)
            if not os.path.isdir(class_path):
                continue
            for fname in sorted(os.listdir(class_path)):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_path, fname))

        print(f'Loaded {len(self.image_paths)} images from {split} split')

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        return self.transform(img)

In [ ]:
train_dataset = ImageNetteDataset('./imagenette2-160', split='train', img_size=128)
val_dataset = ImageNetteDataset('./imagenette2-160', split='val', img_size=128)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')
print(f'Image shape: {train_dataset[0].shape}')
assert train_dataset[0].shape == (3, 128, 128)
assert -1.0 <= train_dataset[0].min() and train_dataset[0].max() <= 1.0

In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False,
    num_workers=2, pin_memory=True
)

In [ ]:
# Визуализация датасета
batch = next(iter(train_loader))
grid = make_grid(batch[:16], nrow=8, normalize=True, value_range=(-1, 1))
fig, ax = plt.subplots(1, 1, figsize=(16, 4))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title('ImageNette 128x128 samples')
ax.axis('off')
plt.tight_layout()
plt.show()

# Часть 1. Архитектура VAE

Мы реализуем свёрточный VAE с архитектурой, приближенной к VAE из Stable Diffusion. Основные компоненты:

- **ResBlock** — основной строительный блок (GroupNorm → SiLU → Conv → GroupNorm → SiLU → Conv + skip-connection)
- **Encoder** — сжимает $128 \times 128 \times 3$ в $16 \times 16 \times 4$ (×8 downsampling, 3 уровня)
- **Decoder** — восстанавливает из $16 \times 16 \times 4$ обратно в $128 \times 128 \times 3$

Каналы по уровням: $[64, 128, 256]$, mid-блок: $256$ каналов.

```
Encoder: 3 → [Conv3x3] → 64 → [ResBlock, ResBlock, ↓] → 128 → [ResBlock, ResBlock, ↓] → 256 → [ResBlock, ResBlock, ↓] → 256 (mid) → [GN + SiLU + Conv] → 8
                128x128              64x64                    32x32                     16x16                 16x16                    16x16

Decoder: 4 → [Conv3x3] → 256 → [ResBlock, ResBlock] (mid) → 256 → [ResBlock, ResBlock, ↑] → 128 → [ResBlock, ResBlock, ↑] → 64 → [ResBlock, ResBlock, ↑] → [GN + SiLU + Conv] → 3
              16x16             16x16                       16x16                      32x32                      64x64                    128x128
```

Обратите внимание: encoder выдаёт 8 каналов (по 4 для $\mu$ и $\log\sigma^2$), а decoder принимает 4 канала (сэмпл из $q(z|x)$).

## 1.1 ResBlock

Реализуйте Residual Block, используемый в VAE архитектурах для LDM:

```
x --> GroupNorm(32) -> SiLU -> Conv3x3 -> GroupNorm(32) -> SiLU -> Conv3x3 --> (+) -> out
  |                                                                              ^
  └──────────────────── [Conv1x1 если in_ch != out_ch] ────────────────────────┘
```

**Важно:**
- Используйте `GroupNorm` с `num_groups=32` (стандарт в SD)
- Активация `SiLU` (она же Swish): $\text{SiLU}(x) = x \cdot \sigma(x)$
- Если `in_channels != out_channels`, нужна проекция через Conv1×1 для skip-connection
- Padding для Conv3×3 — 1 (сохраняем размер)

In [ ]:
class ResBlock(nn.Module):
    """Residual block with GroupNorm and SiLU activation."""

    def __init__(self, in_channels: int, out_channels: int):
        """
        Args:
            in_channels: (int) Number of input channels
            out_channels: (int) Number of output channels
        """
        super().__init__()

        # TODO: реализуйте основную ветку (2 свёртки с GroupNorm и SiLU)
        self.norm1 = ...  # GroupNorm(32, in_channels)
        self.conv1 = ...  # Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = ...  # GroupNorm(32, out_channels)
        self.conv2 = ...  # Conv2d(out_channels, out_channels, 3, padding=1)

        # TODO: реализуйте skip-connection (Identity если каналы совпадают, Conv1x1 иначе)
        if in_channels != out_channels:
            self.skip = ...  # Conv2d(in_channels, out_channels, 1)
        else:
            self.skip = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (torch.Tensor) Input tensor (B, C_in, H, W)

        Returns:
            Output tensor (B, C_out, H, W)
        """
        # TODO: реализуйте forward pass
        # h = norm1 -> silu -> conv1 -> norm2 -> silu -> conv2
        # return h + skip(x)
        raise NotImplementedError

NameError: name 'nn' is not defined

In [ ]:
# Тесты для ResBlock
# Тест 1: одинаковые каналы
rb_same = ResBlock(64, 64)
print(hide_specs(rb_same))
check_numel(rb_same, 74112)

x = torch.randn(2, 64, 16, 16)
out = rb_same(x)
assert out.shape == (2, 64, 16, 16), f'Expected (2, 64, 16, 16), got {out.shape}'

# Тест 2: разные каналы
rb_diff = ResBlock(64, 128)
print(hide_specs(rb_diff))
check_numel(rb_diff, 230144)

x = torch.randn(2, 64, 32, 32)
out = rb_diff(x)
assert out.shape == (2, 128, 32, 32), f'Expected (2, 128, 32, 32), got {out.shape}'

# Тест 3: проверка residual connection
rb_test = ResBlock(64, 64)
with torch.no_grad():
    for p in rb_test.parameters():
        p.zero_()
x = torch.randn(2, 64, 8, 8)
out = rb_test(x)
assert torch.allclose(out, x, atol=1e-6), 'With zero weights, ResBlock should be identity'

print('All ResBlock tests passed!')

## 1.2 Downsample и Upsample

Реализуйте блоки для изменения пространственного размера:

- **Downsample**: свёртка $3 \times 3$ с `stride=2` (уменьшает размер вдвое)
- **Upsample**: nearest-neighbor interpolation ×2, затем свёртка $3 \times 3$ (увеличивает размер вдвое)

В Stable Diffusion используется именно этот подход (вместо пулинга/транспонированных свёрток).

In [ ]:
class Downsample(nn.Module):
    """Spatial downsampling via strided convolution."""

    def __init__(self, channels: int):
        """
        Args: 
            channels: (int) Number of channels (unchanged)
        """
        super().__init__()
        # TODO: Conv2d с stride=2 и padding=1
        self.conv = ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
        x: (torch.Tensor) (B, C, H, W)
        
        Returns:
            (torch.Tensor) (B, C, H//2, W//2)
        """
        # TODO
        raise NotImplementedError


class Upsample(nn.Module):
    """Spatial upsampling via nearest interpolation + convolution."""

    def __init__(self, channels: int):
        """
        Args: 
        channels: (int) Number of channels (unchanged)
        """
        super().__init__()
        # TODO: Conv2d 3x3 с padding=1
        self.conv = ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
        x: (torch.Tensor) (B, C, H, W)
        
        Returns:
         (torch.Tensor) (B, C, 2*H, 2*W)
        """
        # TODO: F.interpolate(scale_factor=2, mode='nearest') -> conv
        raise NotImplementedError

In [ ]:
# Тесты для Downsample/Upsample
down = Downsample(64)
check_numel(down, 36928)
x = torch.randn(2, 64, 32, 32)
assert down(x).shape == (2, 64, 16, 16)

up = Upsample(64)
check_numel(up, 36928)
x = torch.randn(2, 64, 16, 16)
assert up(x).shape == (2, 64, 32, 32)

# Round-trip shape check
x = torch.randn(2, 128, 64, 64)
down128 = Downsample(128)
up128 = Upsample(128)
assert up128(down128(x)).shape == x.shape

print('All Downsample/Upsample tests passed!')

## 1.3 Encoder

Реализуйте Encoder, который сжимает изображение $128 \times 128 \times 3$ в карту параметров $16 \times 16 \times 8$ (по 4 канала для $\mu$ и $\log \sigma^2$).

Архитектура:
1. Входная свёртка: `Conv2d(3, 64, 3, padding=1)` → $128 \times 128 \times 64$
2. **Уровень 1** ($128 \to 64$): `ResBlock(64, 64)` → `ResBlock(64, 64)` → `Downsample(64)` → $64 \times 64 \times 64$
3. **Уровень 2** ($64 \to 128$): `ResBlock(64, 128)` → `ResBlock(128, 128)` → `Downsample(128)` → $32 \times 32 \times 128$
4. **Уровень 3** ($128 \to 256$): `ResBlock(128, 256)` → `ResBlock(256, 256)` → `Downsample(256)` → $16 \times 16 \times 256$
5. **Mid-блок**: `ResBlock(256, 256)` → `ResBlock(256, 256)` → $16 \times 16 \times 256$
6. Выходной слой: `GroupNorm(32, 256)` → `SiLU` → `Conv2d(256, 2*latent_channels, 3, padding=1)` → $16 \times 16 \times 8$

**Замечание:** первый `ResBlock` каждого уровня меняет число каналов (64→64, 64→128, 128→256), второй — сохраняет.

In [ ]:
class Encoder(nn.Module):
    """VAE Encoder: 128x128x3 -> 16x16x(2*latent_channels)."""

    def __init__(self, latent_channels: int = 4, ch: int = 64, ch_mults: Tuple[int, ...] = (1, 2, 4)):
        """
        :param int latent_channels: Number of latent channels (output has 2*latent_channels for mu and logvar)
        :param int ch: Base channel count
        :param tuple ch_mults: Channel multipliers per level
        """
        super().__init__()

        self.latent_channels = latent_channels
        channels = [ch * m for m in ch_mults]  # [64, 128, 256]

        # TODO: реализуйте encoder

        # 1. Входная свёртка
        self.conv_in = ...  # Conv2d(3, ch, 3, padding=1)

        # 2. Блоки по уровням (ResBlock, ResBlock, Downsample) x len(ch_mults)
        self.down_blocks = nn.ModuleList()
        in_ch = ch
        for out_ch in channels:
            self.down_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch),
                ResBlock(out_ch, out_ch),
                Downsample(out_ch),
            ]))
            in_ch = out_ch

        # 3. Mid-блок
        self.mid_block1 = ResBlock(channels[-1], channels[-1])
        self.mid_block2 = ResBlock(channels[-1], channels[-1])

        # 4. Выходной слой
        self.norm_out = ...  # GroupNorm(32, channels[-1])
        self.conv_out = ...  # Conv2d(channels[-1], 2 * latent_channels, 3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        :param torch.Tensor x: Input images (B, 3, 128, 128)
        :return: Parameters (B, 2*latent_channels, 16, 16)
        :rtype: torch.Tensor
        """
        # TODO: реализуйте forward pass
        # h = conv_in(x)
        # for (resblock1, resblock2, downsample) in down_blocks:
        #     h = resblock1(h) -> resblock2(h) -> downsample(h)
        # h = mid_block1(h) -> mid_block2(h)
        # h = silu(norm_out(h))
        # return conv_out(h)
        raise NotImplementedError

In [ ]:
# Тесты для Encoder
encoder = Encoder(latent_channels=4)
print(hide_specs(encoder))

x = torch.randn(2, 3, 128, 128)
out = encoder(x)
assert out.shape == (2, 8, 16, 16), f'Expected (2, 8, 16, 16), got {out.shape}'

# Проверка на другом размере
x_small = torch.randn(2, 3, 64, 64)
out_small = encoder(x_small)
assert out_small.shape == (2, 8, 8, 8), f'Expected (2, 8, 8, 8), got {out_small.shape}'

total_params = sum(p.numel() for p in encoder.parameters())
print(f'Encoder parameters: {total_params:,}')
assert 3_000_000 < total_params < 6_000_000, f'Unexpected parameter count: {total_params}'

print('All Encoder tests passed!')

## 1.4 Decoder

Реализуйте Decoder, который восстанавливает изображение из латентного представления: $16 \times 16 \times 4 \to 128 \times 128 \times 3$.

Архитектура зеркальна Encoder'у:
1. Входная свёртка: `Conv2d(latent_channels, 256, 3, padding=1)` → $16 \times 16 \times 256$
2. **Mid-блок**: `ResBlock(256, 256)` → `ResBlock(256, 256)`
3. **Уровень 3** ($256 \to 256$): `ResBlock(256, 256)` → `ResBlock(256, 256)` → `Upsample(256)` → $32 \times 32 \times 256$
4. **Уровень 2** ($256 \to 128$): `ResBlock(256, 128)` → `ResBlock(128, 128)` → `Upsample(128)` → $64 \times 64 \times 128$
5. **Уровень 1** ($128 \to 64$): `ResBlock(128, 64)` → `ResBlock(64, 64)` → `Upsample(64)` → $128 \times 128 \times 64$
6. Выходной слой: `GroupNorm(32, 64)` → `SiLU` → `Conv2d(64, 3, 3, padding=1)` → $128 \times 128 \times 3$

**Замечание:** в decoder первый `ResBlock` каждого уровня принимает каналы с предыдущего уровня (256→256, 256→128, 128→64).

In [ ]:
class Decoder(nn.Module):
    """VAE Decoder: 16x16 x latent_channels -> 128x128x3."""

    def __init__(self, latent_channels: int = 4, ch: int = 64, ch_mults: Tuple[int, ...] = (1, 2, 4)):
        """
        :param int latent_channels: Number of latent channels
        :param int ch: Base channel count
        :param tuple ch_mults: Channel multipliers per level
        """
        super().__init__()

        channels = [ch * m for m in ch_mults]  # [64, 128, 256]

        # TODO: реализуйте decoder

        # 1. Входная свёртка
        self.conv_in = ...  # Conv2d(latent_channels, channels[-1], 3, padding=1)

        # 2. Mid-блок
        self.mid_block1 = ResBlock(channels[-1], channels[-1])
        self.mid_block2 = ResBlock(channels[-1], channels[-1])

        # 3. Блоки по уровням в обратном порядке (ResBlock, ResBlock, Upsample)
        self.up_blocks = nn.ModuleList()
        in_ch = channels[-1]
        for out_ch in reversed(channels):
            self.up_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch),
                ResBlock(out_ch, out_ch),
                Upsample(out_ch),
            ]))
            in_ch = out_ch

        # 4. Выходной слой
        self.norm_out = ...  # GroupNorm(32, channels[0])
        self.conv_out = ...  # Conv2d(channels[0], 3, 3, padding=1)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        :param torch.Tensor z: Latent tensor (B, latent_channels, 16, 16)
        :return: Reconstructed images (B, 3, 128, 128)
        :rtype: torch.Tensor
        """
        # TODO: реализуйте forward pass
        # h = conv_in(z)
        # h = mid_block1(h) -> mid_block2(h)
        # for (resblock1, resblock2, upsample) in up_blocks:
        #     h = resblock1(h) -> resblock2(h) -> upsample(h)
        # h = silu(norm_out(h))
        # return conv_out(h)
        raise NotImplementedError

In [ ]:
# Тесты для Decoder
decoder = Decoder(latent_channels=4)
print(hide_specs(decoder))

z = torch.randn(2, 4, 16, 16)
out = decoder(z)
assert out.shape == (2, 3, 128, 128), f'Expected (2, 3, 128, 128), got {out.shape}'

total_params = sum(p.numel() for p in decoder.parameters())
print(f'Decoder parameters: {total_params:,}')
assert 3_000_000 < total_params < 6_000_000, f'Unexpected parameter count: {total_params}'

print('All Decoder tests passed!')

## 1.5 VAE: Reparameterization Trick

Объедините Encoder и Decoder в полный VAE. Ключевой момент — **reparameterization trick**:

$$z = \mu + \sigma \odot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

где $\mu$ и $\log \sigma^2$ — выходы Encoder'а (по 4 канала каждый).

**Замечание:** при инференсе (eval mode) мы не добавляем шум и используем $z = \mu$.

In [ ]:
class VAE(nn.Module):
    """Full VAE: Encoder + reparameterization + Decoder."""

    def __init__(self, latent_channels: int = 4, ch: int = 64, ch_mults: Tuple[int, ...] = (1, 2, 4)):
        super().__init__()

        self.latent_channels = latent_channels
        self.latent_h = 16
        self.latent_w = 16

        self.encoder = Encoder(latent_channels, ch, ch_mults)
        self.decoder = Decoder(latent_channels, ch, ch_mults)

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Encode images to latent parameters.
        :param torch.Tensor x: (B, 3, H, W)
        :return: (mu, logvar), each (B, latent_channels, h, w)
        """
        # TODO: пропустить x через encoder, разделить на mu и logvar по каналам
        # h = self.encoder(x)  # (B, 2*latent_channels, h, w)
        # mu, logvar = torch.chunk(h, 2, dim=1)
        raise NotImplementedError

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        Sample z using reparameterization trick.
        :param torch.Tensor mu: Mean (B, C, H, W)
        :param torch.Tensor logvar: Log variance (B, C, H, W)
        :return: Sampled z (B, C, H, W)
        """
        # TODO: реализуйте reparameterization trick
        # В training mode: z = mu + std * eps, где std = exp(0.5 * logvar), eps ~ N(0, I)
        # В eval mode: z = mu
        raise NotImplementedError

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """
        Decode latent to images.
        :param torch.Tensor z: (B, latent_channels, h, w)
        :return: (B, 3, H, W)
        """
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Full forward pass.
        :param torch.Tensor x: (B, 3, H, W)
        :return: (reconstruction, mu, logvar)
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [ ]:
# Тесты для VAE
set_global_seed(42)
vae = VAE(latent_channels=4)

x = torch.randn(2, 3, 128, 128)
recon, mu, logvar = vae(x)

assert recon.shape == (2, 3, 128, 128), f'Reconstruction shape: {recon.shape}'
assert mu.shape == (2, 4, 16, 16), f'Mu shape: {mu.shape}'
assert logvar.shape == (2, 4, 16, 16), f'Logvar shape: {logvar.shape}'

# Проверка reparameterization: в eval mode должен возвращать mu
vae.eval()
recon1, mu1, _ = vae(x)
recon2, mu2, _ = vae(x)
assert torch.allclose(recon1, recon2), 'In eval mode, reconstructions should be deterministic'

# В train mode — стохастический
vae.train()
recon1, _, _ = vae(x)
recon2, _, _ = vae(x)
assert not torch.allclose(recon1, recon2), 'In train mode, reconstructions should be stochastic'

total_params = sum(p.numel() for p in vae.parameters())
print(f'Total VAE parameters: {total_params:,}')

print('All VAE tests passed!')

# Часть 2. Perceptual Loss и обучение

Попиксельные функции потерь (MSE, L1) приводят к размытым реконструкциям, поскольку оптимальный «средний» пиксель для нескольких возможных решений — это их среднее. Perceptual loss решает эту проблему, сравнивая изображения на уровне высокоуровневых признаков VGG сети.

## 2.1 VGG Feature Extractor

Используем предобученный VGG19 для извлечения признаков с нескольких уровней. Входные изображения нормализуются в соответствии со стандартом ImageNet.

In [ ]:
class VGGFeatureExtractor(nn.Module):
    """Extract multi-scale features from pretrained VGG19 for perceptual loss."""

    def __init__(self):
        super().__init__()
        vgg = torchvision.models.vgg19(weights=torchvision.models.VGG19_Weights.IMAGENET1K_V1).features

        # Извлекаем блоки до relu1_2, relu2_2, relu3_4, relu4_4, relu5_4
        self.blocks = nn.ModuleList([
            vgg[:4],   # relu1_2  (64 channels,  HxW)
            vgg[4:9],  # relu2_2  (128 channels, H/2 x W/2)
            vgg[9:18], # relu3_4  (256 channels, H/4 x W/4)
            vgg[18:27],# relu4_4  (512 channels, H/8 x W/8)
            vgg[27:36],# relu5_4  (512 channels, H/16 x W/16)
        ])

        # Замораживаем веса
        for param in self.parameters():
            param.requires_grad = False

        # ImageNet нормализация
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def preprocess(self, x: torch.Tensor) -> torch.Tensor:
        """Преобразование из [-1, 1] в ImageNet-нормализованное пространство."""
        x = (x + 1) / 2  # [-1, 1] -> [0, 1]
        return (x - self.mean) / self.std

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        """
        :param torch.Tensor x: Images in [-1, 1] range (B, 3, H, W)
        :return: List of feature tensors from each VGG block
        """
        x = self.preprocess(x)
        features = []
        for block in self.blocks:
            x = block(x)
            features.append(x)
        return features

In [ ]:
# Тест VGG Feature Extractor
vgg_features = VGGFeatureExtractor().to(device)
x = torch.randn(2, 3, 128, 128).to(device)
feats = vgg_features(x)
print('VGG feature shapes:')
for i, f in enumerate(feats):
    print(f'  Block {i}: {f.shape}')
assert len(feats) == 5
assert feats[0].shape == (2, 64, 128, 128)
assert feats[4].shape == (2, 512, 8, 8)

## 2.2 Perceptual Loss

Реализуйте perceptual loss — взвешенную сумму L1-расстояний между VGG-признаками оригинала и реконструкции:

$$\mathcal{L}_{\text{perc}} = \sum_{l=0}^{4} \frac{1}{C_l H_l W_l} \| \phi_l(x) - \phi_l(\hat{x}) \|_1$$

где $\phi_l$ — признаки $l$-го блока VGG.

In [ ]:
class PerceptualLoss(nn.Module):
    """LPIPS-style perceptual loss using VGG19 features."""

    def __init__(self):
        super().__init__()
        self.vgg = VGGFeatureExtractor()

    def forward(self, x: torch.Tensor, recon: torch.Tensor) -> torch.Tensor:
        """
        Compute perceptual loss between original and reconstructed images.

        :param torch.Tensor x: Original images (B, 3, H, W), range [-1, 1]
        :param torch.Tensor recon: Reconstructed images (B, 3, H, W), range [-1, 1]
        :return: Scalar loss
        :rtype: torch.Tensor
        """
        # TODO: реализуйте perceptual loss
        # 1. Извлеките VGG-фичи для x и recon
        # 2. Для каждого уровня l вычислите L1-расстояние между фичами,
        #    нормализованное на число элементов (C*H*W)
        # 3. Верните сумму расстояний по всем уровням
        raise NotImplementedError

In [ ]:
# Тест Perceptual Loss
perc_loss_fn = PerceptualLoss().to(device)

x = torch.randn(2, 3, 128, 128).to(device)
recon = torch.randn(2, 3, 128, 128).to(device)
loss = perc_loss_fn(x, recon)
assert loss.shape == (), f'Expected scalar, got {loss.shape}'
assert loss.item() > 0, 'Perceptual loss should be positive'

# Одинаковые изображения -> нулевой лосс
loss_same = perc_loss_fn(x, x)
assert loss_same.item() < 1e-6, f'Same images should have ~0 perceptual loss, got {loss_same.item()}'

print(f'Perceptual loss (random): {loss.item():.4f}')
print(f'Perceptual loss (same):   {loss_same.item():.6f}')
print('All PerceptualLoss tests passed!')

## 2.3 Функция потерь и обучение

Реализуйте KL-дивергенцию и полный цикл обучения.

KL-дивергенция для гауссовского VAE:

$$D_{\text{KL}} = \frac{1}{2} \cdot \text{mean}\left( \mu^2 + e^{\log\sigma^2} - \log\sigma^2 - 1 \right)$$

**Рекомендуемые гиперпараметры:**
- $\lambda_{\text{rec}} = 1.0$ (L1 loss), $\lambda_{\text{perc}} = 1.0$, $\beta = 10^{-6}$
- Learning rate: $10^{-4}$ (Adam), batch size: 16
- Epochs: 30–50. Обучение ~30–60 минут на T4.

In [ ]:
def kl_divergence(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """
    Compute KL divergence KL(q(z|x) || p(z)) for Gaussian q and standard normal p.
    Averaged over all dimensions.

    :param torch.Tensor mu: Mean of q(z|x), shape (B, C, H, W)
    :param torch.Tensor logvar: Log variance of q(z|x), shape (B, C, H, W)
    :return: Scalar KL divergence
    """
    # TODO: реализуйте KL дивергенцию
    # KL = 0.5 * mean(mu^2 + exp(logvar) - logvar - 1)
    raise NotImplementedError

In [ ]:
# Тест KL
mu = torch.zeros(4, 4, 16, 16)
logvar = torch.zeros(4, 4, 16, 16)
assert kl_divergence(mu, logvar).item() < 1e-6, 'KL(N(0,1) || N(0,1)) should be 0'

mu = torch.ones(4, 4, 16, 16)
logvar = torch.zeros(4, 4, 16, 16)
expected_kl = 0.5  # KL(N(1,1) || N(0,1)) = 0.5
assert abs(kl_divergence(mu, logvar).item() - expected_kl) < 1e-5, f'Expected {expected_kl}'

print('KL divergence tests passed!')

In [ ]:
config = {
    'lr': 1e-4,
    'batch_size': 16,
    'num_epochs': 40,
    'lambda_rec': 1.0,
    'lambda_perc': 1.0,
    'beta_kl': 1e-6,
    'ema_decay': 0.999,
    'log_interval': 100,
    'val_interval': 5,
}

In [ ]:
def train_vae(config, train_loader, val_loader, device):
    """Train VAE with reconstruction + perceptual + KL loss."""

    vae = VAE(latent_channels=4).to(device)
    perceptual_loss_fn = PerceptualLoss().to(device)
    ema_vae = AveragedModel(vae, multi_avg_fn=get_ema_multi_avg_fn(decay=config['ema_decay']))

    optimizer = torch.optim.Adam(vae.parameters(), lr=config['lr'])

    wandb.init(project='ldm-vae-homework', config=config, name='vae-perceptual')

    scaler = torch.cuda.amp.GradScaler()
    global_step = 0

    for epoch in range(config['num_epochs']):
        vae.train()
        epoch_losses = {'total': [], 'rec': [], 'perc': [], 'kl': []}
        pbar = tqdm.tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["num_epochs"]}')

        for batch in pbar:
            images = batch.to(device)

            with torch.cuda.amp.autocast():
                recon, mu, logvar = vae(images)

                # ================================================
                # TODO: вычислите компоненты функции потерь
                # ================================================
                # 1. Reconstruction loss (L1)
                loss_rec = ...  # F.l1_loss(recon, images)
                # 2. Perceptual loss
                loss_perc = ...  # perceptual_loss_fn(images, recon)
                # 3. KL divergence
                loss_kl = ...  # kl_divergence(mu, logvar)

                # 4. Total loss
                loss = (config['lambda_rec'] * loss_rec
                        + config['lambda_perc'] * loss_perc
                        + config['beta_kl'] * loss_kl)
                # ================================================

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            ema_vae.update_parameters(vae)

            epoch_losses['total'].append(loss.item())
            epoch_losses['rec'].append(loss_rec.item())
            epoch_losses['perc'].append(loss_perc.item())
            epoch_losses['kl'].append(loss_kl.item())

            if global_step % config['log_interval'] == 0:
                wandb.log({
                    'train/loss': loss.item(),
                    'train/rec_loss': loss_rec.item(),
                    'train/perc_loss': loss_perc.item(),
                    'train/kl_loss': loss_kl.item(),
                }, step=global_step)

            pbar.set_postfix(loss=f'{loss.item():.4f}', rec=f'{loss_rec.item():.4f}')
            global_step += 1

        # Валидация
        if (epoch + 1) % config['val_interval'] == 0:
            vae.eval()
            ema_vae.eval()
            val_images = next(iter(val_loader))[:8].to(device)
            with torch.inference_mode():
                val_recon, _, _ = vae(val_images)
                ema_recon, _, _ = ema_vae(val_images)

            for tag, rec in [('val/recon', val_recon), ('val/recon_ema', ema_recon)]:
                comparison = torch.cat([val_images, rec], dim=0)
                grid = make_grid(comparison, nrow=8, normalize=True, value_range=(-1, 1))
                wandb.log({tag: wandb.Image(grid.permute(1, 2, 0).cpu().numpy())}, step=global_step)

        avg = {k: np.mean(v) for k, v in epoch_losses.items()}
        print(f'Epoch {epoch+1}: total={avg["total"]:.4f}, rec={avg["rec"]:.4f}, '
              f'perc={avg["perc"]:.4f}, kl={avg["kl"]:.2f}')

    wandb.finish()
    return vae, ema_vae

In [ ]:
set_global_seed(42)
vae_trained, ema_vae_trained = train_vae(config, train_loader, val_loader, device)

### Визуализация результатов

Посмотрим на реконструкции обученной модели:

In [ ]:
@torch.inference_mode()
def show_reconstructions(model, dataloader, device, n=8, title=''):
    """Visualize original images and their reconstructions."""
    model.eval()
    images = next(iter(dataloader))[:n].to(device)
    recon, _, _ = model(images)

    fig, axes = plt.subplots(2, n, figsize=(n * 2, 4))
    for i in range(n):
        img_orig = images[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
        img_recon = recon[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
        axes[0, i].imshow(np.clip(img_orig, 0, 1))
        axes[1, i].imshow(np.clip(img_recon, 0, 1))
        axes[0, i].axis('off')
        axes[1, i].axis('off')
    axes[0, 0].set_ylabel('Original', fontsize=12)
    axes[1, 0].set_ylabel('Recon', fontsize=12)
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_reconstructions(vae_trained, val_loader, device, title='VAE (Perceptual)')
show_reconstructions(ema_vae_trained, val_loader, device, title='VAE (Perceptual, EMA)')

# Часть 3. Adversarial Training с PatchGAN

Для получения резких, реалистичных текстур в реконструкциях мы добавляем **PatchGAN дискриминатор**. Он обучается отличать «настоящие» патчи изображений от «реконструированных», а VAE учится обманывать дискриминатор.

## 3.1 PatchGAN Discriminator

PatchDiscriminator выдаёт не одно число «real/fake», а **карту** предсказаний — для каждого патча входного изображения отдельное предсказание.

Архитектура:
```
Input (3, 128, 128)
  -> Conv2d(3, 64, 4, stride=2, padding=1) -> LeakyReLU(0.2)                    -> (64, 64, 64)
  -> Conv2d(64, 128, 4, stride=2, padding=1) -> InstanceNorm2d -> LeakyReLU(0.2) -> (128, 32, 32)
  -> Conv2d(128, 256, 4, stride=2, padding=1) -> InstanceNorm2d -> LeakyReLU(0.2)-> (256, 16, 16)
  -> Conv2d(256, 1, 4, padding=1)                                                -> (1, 15, 15)
```

In [ ]:
class PatchDiscriminator(nn.Module):
    """PatchGAN discriminator for adversarial training."""

    def __init__(self, in_channels: int = 3):
        """
        :param int in_channels: Number of input image channels
        """
        super().__init__()

        # TODO: реализуйте PatchDiscriminator
        # 4 слоя:
        #   1. Conv2d(in_channels, 64, 4, stride=2, padding=1) -> LeakyReLU(0.2)
        #   2. Conv2d(64, 128, 4, stride=2, padding=1) -> InstanceNorm2d(128) -> LeakyReLU(0.2)
        #   3. Conv2d(128, 256, 4, stride=2, padding=1) -> InstanceNorm2d(256) -> LeakyReLU(0.2)
        #   4. Conv2d(256, 1, 4, padding=1)
        self.model = ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        :param torch.Tensor x: Input images (B, 3, H, W)
        :return: Patch predictions (B, 1, H', W')
        :rtype: torch.Tensor
        """
        # TODO
        raise NotImplementedError

In [ ]:
# Тесты для PatchDiscriminator
disc = PatchDiscriminator()
print(hide_specs(disc))

x = torch.randn(2, 3, 128, 128)
out = disc(x)
assert out.shape == (2, 1, 15, 15), f'Expected (2, 1, 15, 15), got {out.shape}'

total_params = sum(p.numel() for p in disc.parameters())
print(f'Discriminator parameters: {total_params:,}')

print('PatchDiscriminator tests passed!')

## 3.2 Hinge Loss

Для обучения GAN используется **hinge loss**:

**Дискриминатор:**
$$\mathcal{L}_D = \mathbb{E}\big[\text{ReLU}(1 - D(x))\big] + \mathbb{E}\big[\text{ReLU}(1 + D(\hat{x}))\big]$$

**Генератор (VAE):**
$$\mathcal{L}_G^{\text{adv}} = -\mathbb{E}\big[D(\hat{x})\big]$$

In [ ]:
def hinge_loss_discriminator(real_preds: torch.Tensor, fake_preds: torch.Tensor) -> torch.Tensor:
    """
    Compute hinge loss for discriminator.

    :param torch.Tensor real_preds: D(x) predictions on real images
    :param torch.Tensor fake_preds: D(x_hat) predictions on fake images
    :return: Scalar discriminator loss
    """
    # TODO
    raise NotImplementedError


def hinge_loss_generator(fake_preds: torch.Tensor) -> torch.Tensor:
    """
    Compute hinge loss for generator (VAE).

    :param torch.Tensor fake_preds: D(x_hat) predictions on reconstructed images
    :return: Scalar generator adversarial loss
    """
    # TODO
    raise NotImplementedError

In [ ]:
# Тесты для hinge loss
real_preds = torch.tensor([2.0, 0.5, -0.3])
fake_preds = torch.tensor([-2.0, -0.5, 0.3])

d_loss = hinge_loss_discriminator(real_preds, fake_preds)
# ReLU(1-2)=0, ReLU(1-0.5)=0.5, ReLU(1+0.3)=1.3 -> mean real part = 0.6
# ReLU(1+(-2))=0, ReLU(1+(-0.5))=0.5, ReLU(1+0.3)=1.3 -> mean fake part = 0.6
# Total: 1.2
assert abs(d_loss.item() - 1.2) < 1e-5, f'Expected ~1.2, got {d_loss.item()}'

g_loss = hinge_loss_generator(fake_preds)
# -mean(-2, -0.5, 0.3) = 0.7333...
assert abs(g_loss.item() - 0.7333) < 1e-3, f'Expected ~0.733, got {g_loss.item()}'

print('Hinge loss tests passed!')

## 3.3 Цикл обучения VAE + GAN

Цикл обучения чередует шаги дискриминатора и генератора (VAE):

1. **Шаг D:** замораживаем VAE, обучаем D отличать реальные изображения от реконструкций
2. **Шаг G (VAE):** замораживаем D, обучаем VAE с полным лоссом $\mathcal{L}_{\text{rec}} + \mathcal{L}_{\text{perc}} + \beta \cdot D_{\text{KL}} + \lambda_{\text{adv}} \cdot \mathcal{L}_G^{\text{adv}}$

**Важно:** дискриминатор начинает обучаться **после нескольких эпох прогрева** VAE (`disc_start_epoch`), чтобы VAE уже выдавал осмысленные реконструкции.

**Tips:**
- При обучении D используйте `.detach()` для реконструкций
- Можно инициализировать VAE весами из Части 2

In [ ]:
config_gan = {
    'lr_g': 1e-4,
    'lr_d': 1e-4,
    'batch_size': 16,
    'num_epochs': 50,
    'lambda_rec': 1.0,
    'lambda_perc': 1.0,
    'beta_kl': 1e-6,
    'lambda_adv': 0.5,
    'disc_start_epoch': 5,
    'ema_decay': 0.999,
    'log_interval': 100,
    'val_interval': 5,
}

In [ ]:
def train_vae_gan(config, train_loader, val_loader, device, pretrained_vae=None):
    """Train VAE with reconstruction + perceptual + KL + adversarial loss."""

    vae = VAE(latent_channels=4).to(device)
    if pretrained_vae is not None:
        vae.load_state_dict(pretrained_vae.state_dict())
        print('Loaded pretrained VAE weights')

    disc = PatchDiscriminator().to(device)
    perceptual_loss_fn = PerceptualLoss().to(device)
    ema_vae = AveragedModel(vae, multi_avg_fn=get_ema_multi_avg_fn(decay=config['ema_decay']))

    optimizer_g = torch.optim.Adam(vae.parameters(), lr=config['lr_g'], betas=(0.5, 0.9))
    optimizer_d = torch.optim.Adam(disc.parameters(), lr=config['lr_d'], betas=(0.5, 0.9))

    scaler = torch.cuda.amp.GradScaler()

    wandb.init(project='ldm-vae-homework', config=config, name='vae-gan')

    global_step = 0

    for epoch in range(config['num_epochs']):
        vae.train()
        disc.train()
        use_disc = epoch >= config['disc_start_epoch']

        epoch_losses = {'g': [], 'rec': [], 'perc': [], 'kl': [], 'g_adv': [], 'd': []}
        pbar = tqdm.tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["num_epochs"]}')

        for batch in pbar:
            images = batch.to(device)

            # ================================================
            # ШАГ 1: Обучение дискриминатора
            # ================================================
            if use_disc:
                with torch.cuda.amp.autocast():
                    with torch.no_grad():
                        recon_d, _, _ = vae(images)
                        recon_d = recon_d.detach()

                    # TODO: предсказания D на реальных и реконструированных изображениях
                    real_preds = ...  # disc(images)
                    fake_preds = ...  # disc(recon_d)

                    # TODO: hinge loss для дискриминатора
                    d_loss = ...  # hinge_loss_discriminator(real_preds, fake_preds)

                optimizer_d.zero_grad()
                scaler.scale(d_loss).backward()
                scaler.unscale_(optimizer_d)
                torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
                scaler.step(optimizer_d)
            else:
                d_loss = torch.tensor(0.0)

            # ================================================
            # ШАГ 2: Обучение генератора (VAE)
            # ================================================
            with torch.cuda.amp.autocast():
                recon, mu, logvar = vae(images)

                loss_rec = F.l1_loss(recon, images)
                loss_perc = perceptual_loss_fn(images, recon)
                loss_kl = kl_divergence(mu, logvar)

                loss_g = (config['lambda_rec'] * loss_rec
                          + config['lambda_perc'] * loss_perc
                          + config['beta_kl'] * loss_kl)

                # TODO: если дискриминатор активен, добавьте adversarial loss
                if use_disc:
                    fake_preds_g = ...  # disc(recon)
                    loss_adv = ...  # hinge_loss_generator(fake_preds_g)
                    loss_g = loss_g + config['lambda_adv'] * loss_adv
                else:
                    loss_adv = torch.tensor(0.0)

            optimizer_g.zero_grad()
            scaler.scale(loss_g).backward()
            scaler.unscale_(optimizer_g)
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            scaler.step(optimizer_g)

            scaler.update()
            ema_vae.update_parameters(vae)

            epoch_losses['g'].append(loss_g.item())
            epoch_losses['rec'].append(loss_rec.item())
            epoch_losses['perc'].append(loss_perc.item())
            epoch_losses['kl'].append(loss_kl.item())
            epoch_losses['g_adv'].append(loss_adv.item() if torch.is_tensor(loss_adv) else 0)
            epoch_losses['d'].append(d_loss.item() if torch.is_tensor(d_loss) else 0)

            if global_step % config['log_interval'] == 0:
                wandb.log({
                    'train/loss_g': loss_g.item(),
                    'train/loss_d': d_loss.item() if torch.is_tensor(d_loss) else 0,
                    'train/rec': loss_rec.item(),
                    'train/perc': loss_perc.item(),
                    'train/kl': loss_kl.item(),
                    'train/adv': loss_adv.item() if torch.is_tensor(loss_adv) else 0,
                }, step=global_step)

            pbar.set_postfix(
                g=f'{loss_g.item():.3f}',
                d=f'{d_loss.item():.3f}' if use_disc else 'off',
                rec=f'{loss_rec.item():.3f}'
            )
            global_step += 1

        # Валидация
        if (epoch + 1) % config['val_interval'] == 0:
            vae.eval()
            ema_vae.eval()
            val_images = next(iter(val_loader))[:8].to(device)
            with torch.inference_mode():
                val_recon, _, _ = vae(val_images)
                ema_recon, _, _ = ema_vae(val_images)
            for tag, rec in [('val/recon', val_recon), ('val/recon_ema', ema_recon)]:
                comparison = torch.cat([val_images, rec], dim=0)
                grid = make_grid(comparison, nrow=8, normalize=True, value_range=(-1, 1))
                wandb.log({tag: wandb.Image(grid.permute(1, 2, 0).cpu().numpy())}, step=global_step)

        avg = {k: np.mean(v) for k, v in epoch_losses.items()}
        disc_str = f', d={avg["d"]:.4f}, adv={avg["g_adv"]:.4f}' if use_disc else ''
        print(f'Epoch {epoch+1}: g={avg["g"]:.4f}, rec={avg["rec"]:.4f}, '
              f'perc={avg["perc"]:.4f}, kl={avg["kl"]:.2f}{disc_str}')

    wandb.finish()
    return vae, ema_vae, disc

In [ ]:
set_global_seed(42)

# Можно передать pretrained VAE из части 2 для инициализации
vae_gan, ema_vae_gan, disc_trained = train_vae_gan(
    config_gan, train_loader, val_loader, device,
    pretrained_vae=vae_trained  # или None если хотите обучить с нуля
)

### Сравнение результатов

Сравним реконструкции: без GAN (часть 2) vs с GAN (часть 3). Обратите внимание на резкость текстур:

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(16, 6))

val_images = next(iter(val_loader))[:8].to(device)

with torch.inference_mode():
    recon_perc, _, _ = ema_vae_trained(val_images)
    recon_gan, _, _ = ema_vae_gan(val_images)

for i in range(8):
    for row, imgs, label in [
        (0, val_images, 'Original'),
        (1, recon_perc, 'Perceptual'),
        (2, recon_gan, 'Perceptual + GAN'),
    ]:
        img = imgs[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
        axes[row, i].imshow(np.clip(img, 0, 1))
        axes[row, i].axis('off')
    if i == 0:
        axes[0, 0].set_ylabel('Original', fontsize=11)
        axes[1, 0].set_ylabel('Perc only', fontsize=11)
        axes[2, 0].set_ylabel('Perc+GAN', fontsize=11)

fig.suptitle('Perceptual vs Perceptual+GAN', fontsize=14)
plt.tight_layout()
plt.show()

# Часть 4. Анализ латентного пространства

Финальная часть посвящена анализу свойств обученного латентного пространства. Это важно, потому что в следующем задании мы будем обучать диффузионную модель в этом пространстве!

## 4.1 Интерполяция в латентном пространстве

Линейная интерполяция: $z_\alpha = (1 - \alpha) \cdot z_1 + \alpha \cdot z_2$, $\alpha \in [0, 1]$.

Если пространство гладкое, промежуточные точки должны давать осмысленные изображения.

In [ ]:
@torch.inference_mode()
def visualize_interpolation(model, dataloader, device, n_steps=10):
    """Interpolate between pairs of images in latent space."""
    model.eval()
    images = next(iter(dataloader))[:2].to(device)

    mu1, _ = model.encode(images[0:1])
    mu2, _ = model.encode(images[1:2])

    alphas = torch.linspace(0, 1, n_steps).to(device)
    interpolated = []
    for alpha in alphas:
        z = (1 - alpha) * mu1 + alpha * mu2
        recon = model.decode(z)
        interpolated.append(recon)

    interpolated = torch.cat(interpolated, dim=0)
    grid = make_grid(interpolated, nrow=n_steps, normalize=True, value_range=(-1, 1))

    fig, ax = plt.subplots(1, 1, figsize=(n_steps * 2, 2.5))
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
    ax.set_title('Latent space interpolation (img1 -> img2)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

visualize_interpolation(ema_vae_gan, val_loader, device, n_steps=10)

## 4.2 Статистики латентного пространства

Для диффузионной модели важно, чтобы латентное пространство имело предсказуемую статистику. Визуализируем распределение $\mu$ и $\sigma$ по валидационному датасету:

In [ ]:
@torch.inference_mode()
def analyze_latent_statistics(model, dataloader, device, max_batches=50):
    """Analyze statistics of the latent space."""
    model.eval()
    all_mu = []
    all_std = []

    for i, batch in enumerate(dataloader):
        if i >= max_batches:
            break
        images = batch.to(device)
        mu, logvar = model.encode(images)
        all_mu.append(mu.cpu())
        all_std.append((0.5 * logvar).exp().cpu())

    all_mu = torch.cat(all_mu, dim=0)
    all_std = torch.cat(all_std, dim=0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(all_mu.flatten().numpy(), bins=100, density=True, alpha=0.7)
    axes[0].set_title(f'Distribution of mu\nmean={all_mu.mean():.3f}, std={all_mu.std():.3f}')
    axes[0].set_xlabel('mu values')

    axes[1].hist(all_std.flatten().numpy(), bins=100, density=True, alpha=0.7, color='orange')
    axes[1].set_title(f'Distribution of sigma\nmean={all_std.mean():.3f}, std={all_std.std():.3f}')
    axes[1].set_xlabel('sigma values')

    channel_mu_means = all_mu.mean(dim=(0, 2, 3))
    channel_std_means = all_std.mean(dim=(0, 2, 3))
    x = np.arange(all_mu.shape[1])
    axes[2].bar(x - 0.15, channel_mu_means.numpy(), 0.3, label='E[mu]', alpha=0.7)
    axes[2].bar(x + 0.15, channel_std_means.numpy(), 0.3, label='E[sigma]', alpha=0.7, color='orange')
    axes[2].set_xlabel('Latent channel')
    axes[2].set_title('Per-channel latent statistics')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print(f'Overall: mu mean={all_mu.mean():.4f}, mu std={all_mu.std():.4f}')
    print(f'Overall: sigma mean={all_std.mean():.4f}, sigma std={all_std.std():.4f}')

analyze_latent_statistics(ema_vae_gan, val_loader, device)

## 4.3 Робастность к шуму в латентном пространстве

Критически важно для диффузии: как decoder реагирует на зашумлённые латенты? В процессе обратной диффузии мы итеративно «очищаем» шумные латенты, поэтому decoder должен давать осмысленные результаты даже для слегка зашумлённых $z$.

Визуализируем реконструкции для $z_{\text{noisy}} = z + \sigma_{\text{noise}} \cdot \varepsilon$:

In [ ]:
@torch.inference_mode()
def visualize_noisy_reconstructions(model, dataloader, device, noise_levels=(0, 0.1, 0.3, 0.5, 1.0, 2.0)):
    """Reconstruct images from noise-corrupted latent codes."""
    model.eval()
    images = next(iter(dataloader))[:4].to(device)
    mu, _ = model.encode(images)

    n_imgs = images.shape[0]
    n_levels = len(noise_levels)

    fig, axes = plt.subplots(n_imgs, n_levels + 1, figsize=((n_levels + 1) * 2, n_imgs * 2))

    for i in range(n_imgs):
        img = images[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
        axes[i, 0].imshow(np.clip(img, 0, 1))
        axes[i, 0].set_title('Original' if i == 0 else '')
        axes[i, 0].axis('off')

        for j, sigma in enumerate(noise_levels):
            noise = torch.randn_like(mu[i:i+1]) * sigma
            z_noisy = mu[i:i+1] + noise
            recon = model.decode(z_noisy)
            img_recon = recon[0].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
            axes[i, j + 1].imshow(np.clip(img_recon, 0, 1))
            axes[i, j + 1].set_title(f'noise={sigma}' if i == 0 else '')
            axes[i, j + 1].axis('off')

    fig.suptitle('Decoder robustness to latent noise (important for diffusion!)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

visualize_noisy_reconstructions(ema_vae_gan, val_loader, device)

## Выводы

Ответьте на вопросы:

1. Как изменились реконструкции при переходе от Perceptual к Perceptual+GAN?
2. Какие статистики имеет латентное пространство? Насколько оно близко к $\mathcal{N}(0, I)$?
3. Как decoder справляется с зашумлёнными латентами? Начиная с какого уровня шума реконструкции становятся неузнаваемыми?
4. Почему для латентных диффузий важно, чтобы decoder был робастным к шуму?

# Сохранение модели

Сохраните обученную модель для использования в следующем задании (латентная диффузия):

In [ ]:
# Сохранение EMA весов VAE (лучшая модель)
torch.save({
    'vae_state_dict': ema_vae_gan.module.state_dict(),
    'config': {
        'latent_channels': 4,
        'ch': 64,
        'ch_mults': (1, 2, 4),
        'img_size': 128,
        'latent_size': 16,
    }
}, 'vae_for_ldm.pt')
print('VAE saved to vae_for_ldm.pt')
print('Latent space: 128x128x3 -> 16x16x4 (compression ratio: 48x)')